In [17]:
import pandas as pd
import numpy as np
import subprocess
import sys
import mlflow

In [18]:
# Carregando o dataset tratado para o repositório
df = pd.read_parquet('../data/processed/dataset_tratado.parquet')

In [19]:
import mlflow

# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment("Projeto_Churn_Tech_Challenge")


<Experiment: artifact_location=('file:c:/Users/lara-/OneDrive/Desktop/POS TECH '
 'ML/projeto-tech-challenge-churn/notebooks/mlruns/1'), creation_time=1774800663369, experiment_id='1', last_update_time=1774800663369, lifecycle_stage='active', name='Projeto_Churn_Tech_Challenge', tags={}, workspace='default'>

In [15]:
mlflow.get_tracking_uri()

'sqlite:///../mlflow.db'

# --------------- REGRESSÃO LOGISTICA ---------------

In [20]:
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.data
import matplotlib.pyplot as plt # Adicionado para o gráfico
import os # Adicionado para gerenciar arquivos
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                             recall_score, roc_auc_score, confusion_matrix, 
                             classification_report, ConfusionMatrixDisplay) # Adicionado ConfusionMatrixDisplay

# 1. Configurações de separação
test_size = 0.2
random_state = 42

# Separando variáveis alvo e features
X = df.drop(columns=['churn_value'])
y = df['churn_value']

# Split com estratificação (importante para Churn!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)

# 2. Criando o objeto de dataset do MLflow (para a aba "Inputs")
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# Preprocessamento
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# Aplicando transformação
X_train_transformado = preprocessador.fit_transform(X_train)
X_test_transformado = preprocessador.transform(X_test)

feature_names = preprocessador.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_transformado, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformado, columns=feature_names)

# --- INÍCIO DO EXPERIMENTO NO MLFLOW ---
# Nome amigável para a sua Run
run_name_input = "regressao_logistica_baseline"

with mlflow.start_run(run_name=run_name_input) as run:
    
    # Definindo o modelo
    model = LogisticRegression(random_state=random_state, max_iter=1000)
    
    # 1. Logando o Dataset (Aba Inputs)
    mlflow.log_input(dataset, context="training")
    
    # 2. Logando Parâmetros (Aba Parameters)
    mlflow.log_params({
        "model_type": "LogisticRegression",
        "max_iter": 1000,
        "random_state": random_state,
        "test_size": test_size,
        "stratify": "True",
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # 3. Treinando o modelo
    model.fit(X_train_df, y_train)

    # 4. Logando o modelo treinado (Aba Artifacts)
    # É melhor logar DEPOIS do fit para salvar os pesos
    mlflow.sklearn.log_model(model, "logistic_regression_model")

    # Predições e métricas
    y_pred_test = model.predict(X_test_df)
    y_probs = model.predict_proba(X_test_df)[:, 1]

    # Matriz de Confusão numérica
    cm = confusion_matrix(y_test, y_pred_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred_test),
        "precision": precision_score(y_test, y_pred_test),
        "recall": recall_score(y_test, y_pred_test),
        "f1_score": f1_score(y_test, y_pred_test),
        "roc_auc": roc_auc_score(y_test, y_probs)
    }

    # 5. Logando Métricas (Aba Metrics)
    mlflow.log_metrics(metrics)

    # 6. Gerando e Logando o Gráfico da Matriz de Confusão (Aba Artifacts)
    # Define o nome do arquivo temporário
    plot_file_name = "confusion_matrix.png"
    
    # Gera o gráfico
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn']).plot(ax=ax, cmap='Blues')
    ax.set_title(f'Matriz de Confusão - {run_name_input}')
    
    # Salva o gráfico localmente
    plt.savefig(plot_file_name)
    plt.close(fig) # Fecha o plot para liberar memória
    
    # Salva o gráfico como artefato no MLflow
    mlflow.log_artifact(plot_file_name)
    
    # (Opcional) Remove o arquivo temporário local
    if os.path.exists(plot_file_name):
        os.remove(plot_file_name)

    # Output no console
    print(f"Logistic Regression - Métricas de validação")
    print("-" * 30)
    for k, v in metrics.items():
        print(f"{k.capitalize()}: {v:.4f}")
    print("-" * 30)
    print("Confusion Matrix (Numeric):\n", cm)
    print("Classification Report:\n", classification_report(y_test, y_pred_test))

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest

Logistic Regression - Métricas de validação
------------------------------
Accuracy: 0.9688
Precision: 1.0000
Recall: 0.8824
F1_score: 0.9375
Roc_auc: 0.9805
------------------------------
Confusion Matrix (Numeric):
 [[1035    0]
 [  44  330]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      1035
           1       1.00      0.88      0.94       374

    accuracy                           0.97      1409
   macro avg       0.98      0.94      0.96      1409
weighted avg       0.97      0.97      0.97      1409



# --------------- DUMMY CLASSIFIER ---------------

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix


# Criando o objeto de dataset do MLflow (para a aba "Inputs")
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# =========================
# Configurações e Split
# =========================
SEED = 20
test_size = 0.2
X = df.drop(columns=['churn_value'])
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42, stratify=y
)

# =========================
# Preprocessamento
# =========================
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# Aplicando transformação nos dados para o fit manual (fora do pipeline se necessário)
X_train_transformed = preprocessador.fit_transform(X_train)
X_test_transformed = preprocessador.transform(X_test)

# =========================
# Início do MLflow
# =========================
with mlflow.start_run(run_name="Dummy_Classifier_baseline"):

    # 1. Definindo o Modelo
    modelo_dummy = DummyClassifier(strategy='stratified', random_state=SEED)

    mlflow.log_input(dataset, context="training")  # Logando o dataset na aba Inputs

    # 2. Log de Parâmetros
    mlflow.log_params({
        "model_type": "DummyClassifier",
        "random_state": SEED,
        "test_size": test_size,
        "stratify": "True",
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # 3. Treino (Fit)
    # Importante: o fit deve vir antes do log_model para salvar o estado do modelo
    modelo_dummy.fit(X_train_transformed, y_train)

    # 4. Log do Modelo
    mlflow.sklearn.log_model(modelo_dummy, "dummy_classifier_model")

    # 5. Predição e ajuste de Limiar (Exemplo com 0.5 padrão)
    threshold = 0.5
    y_probs = modelo_dummy.predict_proba(X_test_transformed)[:, 1]
    y_pred = (y_probs >= threshold).astype(int)

    # 6. Cálculo e Log de Métricas (Média da Validação Cruzada)
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    
    # Criamos um pipeline apenas para a validação cruzada para evitar data leakage
    from sklearn.pipeline import Pipeline
    pipeline_dummy = Pipeline([('pre', preprocessador), ('model', modelo_dummy)])
    
    resultados_cv = cross_validate(
        pipeline_dummy, X_train, y_train, cv=cv, 
        scoring=['accuracy', 'precision', 'recall', 'f1']
    )

    metrics = {
        "accuracy": resultados_cv['test_accuracy'].mean(),
        "precision": resultados_cv['test_precision'].mean(),
        "recall": resultados_cv['test_recall'].mean(),
        "f1_score": resultados_cv['test_f1'].mean()
    }
    mlflow.log_metrics(metrics)

    # 7. Matriz de Confusão (Artefato)
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn']).plot(ax=ax, cmap='Greys')
    ax.set_title('Matriz de Confusão - Dummy Classifier')
    
    plot_path = "confusion_matrix_dummy.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()

    print("===== Dummy Classifier Gravado no MLflow =====")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest

===== Dummy Classifier Gravado no MLflow =====
accuracy: 0.6006
precision: 0.2624
recall: 0.2789
f1_score: 0.2704


# --------------- REGRESSÃO LINEAR ---------------

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

with mlflow.start_run(run_name="linear_regression"):
    lr_model = LinearRegression()
    mlflow.log_param("model_type", "LinearRegression")

    lr_model.fit(X_train_transformado, y_train)

    y_pred_train_lr = lr_model.predict(X_train_transformado)
    y_pred_test_lr = lr_model.predict(X_test_transformado)

    mse_train = mean_squared_error(y_train, y_pred_train_lr)
    mse_test = mean_squared_error(y_test, y_pred_test_lr)
    rmse_train = np.sqrt(mse_train)
    rmse_test = np.sqrt(mse_test)
    mae_train = mean_absolute_error(y_train, y_pred_train_lr)
    mae_test = mean_absolute_error(y_test, y_pred_test_lr)
    r2_train = r2_score(y_train, y_pred_train_lr)
    r2_test = r2_score(y_test, y_pred_test_lr)

    mlflow.log_metrics({
        "mse_train": mse_train,
        "mse_test": mse_test,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "mae_train": mae_train,
        "mae_test": mae_test,
        "r2_train": r2_train,
        "r2_test": r2_test
    })

    mlflow.sklearn.log_model(lr_model, "linear_regression_model")

    print("Linear Regression - Métricas")
    print(f"RMSE Treino: {rmse_train:.4f}")
    print(f"RMSE Teste: {rmse_test:.4f}")
    print(f"MSE Treino: {mse_train:.4f}")
    print(f"MSE Teste: {mse_test:.4f}")
    print(f"MAE Treino: {mae_train:.4f}")
    print(f"MAE Teste: {mae_test:.4f}")
    print(f"R² Treino: {r2_train:.4f}")
    print(f"R² Teste: {r2_test:.4f}")

# --------------- ÁRVORE DE DECISÃO ---------------

In [22]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                             recall_score, roc_auc_score, confusion_matrix, 
                             classification_report, ConfusionMatrixDisplay)

# --- CONFIGURAÇÕES GERAIS ---
run_name_dt = "decision_tree_baseline   "
max_depth = 5 
random_state = 42
test_size = 0.2

# 1. Preparação dos Dados
X = df.drop(columns=['churn_value'])
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)

# 2. Configuração do Tracking de Dados no MLflow
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# 3. Preprocessamento
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

X_train_transformado = preprocessador.fit_transform(X_train)
X_test_transformado = preprocessador.transform(X_test)

feature_names = preprocessador.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_transformado, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformado, columns=feature_names)

# --- EXECUÇÃO DO EXPERIMENTO ---
with mlflow.start_run(run_name=run_name_dt) as run:
    
    # 1. Definição e Treino
    model_dt = DecisionTreeClassifier(max_depth=max_depth, random_state=random_state)
    model_dt.fit(X_train_df, y_train)
    
    # 2. Log de Inputs e Parâmetros
    mlflow.log_input(dataset, context="training")
    mlflow.log_params({
        "model_type": "DecisionTreeClassifier",
        "max_depth": max_depth,
        "random_state": random_state,
        "test_size": test_size,
        "stratify": "True",
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # 3. Log do Modelo (Artefato)
    mlflow.sklearn.log_model(model_dt, "decision_tree_model")

    # 4. Predições e Métricas
    y_pred_dt = model_dt.predict(X_test_df)
    y_probs_dt = model_dt.predict_proba(X_test_df)[:, 1]

    metrics_dt = {
        "accuracy": accuracy_score(y_test, y_pred_dt),
        "precision": precision_score(y_test, y_pred_dt),
        "recall": recall_score(y_test, y_pred_dt),
        "f1_score": f1_score(y_test, y_pred_dt),
        "roc_auc": roc_auc_score(y_test, y_probs_dt)
    }
    mlflow.log_metrics(metrics_dt)

    # 5. Matriz de Confusão (Gráfico - Artefato)
    cm_dt = confusion_matrix(y_test, y_pred_dt)
    plot_cm_path = "confusion_matrix_dt.png"
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=['No Churn', 'Churn']).plot(ax=ax, cmap='Greens')
    ax.set_title(f'Matriz de Confusão - {run_name_dt}')
    
    plt.savefig(plot_cm_path)
    plt.close(fig)
    mlflow.log_artifact(plot_cm_path)
    if os.path.exists(plot_cm_path): os.remove(plot_cm_path)

    # 6. Feature Importance (Gráfico - Artefato)
    # Extrai as importâncias e associa aos nomes das features
    importances = model_dt.feature_importances_
    # Criamos um DataFrame para ordenar as TOP features
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values(by='importance', ascending=False)
    
    # Pegamos apenas as 15 mais importantes para não poluir o gráfico
    top_features = feature_importance_df.head(15)
    
    # Gera o gráfico de barras horizontais
    plot_fi_path = "feature_importance_dt.png"
    fig_fi, ax_fi = plt.subplots(figsize=(10, 8))
    # Inverte a ordem para a mais importante ficar no topo
    ax_fi.barh(top_features['feature'][::-1], top_features['importance'][::-1], color='seagreen')
    ax_fi.set_xlabel('Importância (Ponderada)')
    ax_fi.set_title(f'Top 15 Variáveis Mais Decisivas para o Churn - {run_name_dt}')
    ax_fi.grid(axis='x', linestyle='--', alpha=0.5)
    
    plt.savefig(plot_fi_path)
    plt.close(fig_fi)
    
    # Loga o gráfico como artefato
    mlflow.log_artifact(plot_fi_path)
    if os.path.exists(plot_fi_path): os.remove(plot_fi_path)

    # (Opcional) Logar a tabela de importância como CSV
    csv_fi_path = "feature_importance_table.csv"
    feature_importance_df.to_csv(csv_fi_path, index=False)
    mlflow.log_artifact(csv_fi_path)
    if os.path.exists(csv_fi_path): os.remove(csv_fi_path)

    # 7. Relatório Final
    print(f"Decision Tree - Rastreamento finalizado no MLflow.")
    print("-" * 30)
    print("Métricas:")
    for k, v in metrics_dt.items():
        print(f"  {k.capitalize()}: {v:.4f}")
    print("-" * 30)
    print("TOP 5 Variáveis Mais Importantes:")
    print(feature_importance_df.head(5))

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest

Decision Tree - Rastreamento finalizado no MLflow.
------------------------------
Métricas:
  Accuracy: 0.9681
  Precision: 0.9853
  Recall: 0.8930
  F1_score: 0.9369
  Roc_auc: 0.9775
------------------------------
TOP 5 Variáveis Mais Importantes:
                                           feature  importance
1173  cat__churn_reason_Attitude of support person    0.976286
1193                            num__tenure_months    0.004678
1194                          num__monthly_charges    0.004365
1143             cat__internet_service_Fiber optic    0.004223
1163                  cat__contract_Month-to-month    0.002492


# --------------- RANDOM FOREST ---------------

In [23]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
from sklearn.ensemble import RandomForestClassifier # O novo modelo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                             recall_score, roc_auc_score, confusion_matrix, 
                             ConfusionMatrixDisplay)

# --- CONFIGURAÇÕES DO RANDOM FOREST ---
run_name_rf = "random_forest_baseline"
n_estimators = 100  # Número de árvores na floresta
max_depth = 10      # Profundidade para evitar overfitting
random_state = 42
test_size = 0.2

# 1. Preparação dos Dados (Mantendo a consistência)
X = df.drop(columns=['churn_value'])
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

# 2. Tracking do Dataset
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# 3. Preprocessamento
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

X_train_transformado = preprocessador.fit_transform(X_train)
X_test_transformado = preprocessador.transform(X_test)
feature_names = preprocessador.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_transformado, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformado, columns=feature_names)

# --- EXECUÇÃO DO EXPERIMENTO ---
with mlflow.start_run(run_name=run_name_rf) as run:
    
    # 1. Definição do Modelo (Random Forest)
    model_rf = RandomForestClassifier(
        n_estimators=n_estimators, 
        max_depth=max_depth, 
        random_state=random_state,
        n_jobs=-1 # Usa todos os núcleos do seu PC para treinar mais rápido
    )
    
    model_rf.fit(X_train_df, y_train)
    
    # 2. Log de Parâmetros Específicos do RF
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "random_state": random_state,
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # 3. Log do Modelo
    mlflow.sklearn.log_model(model_rf, "random_forest_model")

    # 4. Predições e Métricas
    y_pred_rf = model_rf.predict(X_test_df)
    y_probs_rf = model_rf.predict_proba(X_test_df)[:, 1]

    metrics_rf = {
        "accuracy": accuracy_score(y_test, y_pred_rf),
        "precision": precision_score(y_test, y_pred_rf),
        "recall": recall_score(y_test, y_pred_rf),
        "f1_score": f1_score(y_test, y_pred_rf),
        "roc_auc": roc_auc_score(y_test, y_probs_rf)
    }
    mlflow.log_metrics(metrics_rf)

    # 5. Artefato: Matriz de Confusão
    cm_rf = confusion_matrix(y_test, y_pred_rf)
    plot_cm_path = "confusion_matrix_rf.png"
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['No Churn', 'Churn']).plot(ax=ax, cmap='Purples')
    ax.set_title(f'Matriz de Confusão - {run_name_rf}')
    plt.savefig(plot_cm_path)
    plt.close()
    mlflow.log_artifact(plot_cm_path)

    # 6. Artefato: Feature Importance
    importances = model_rf.feature_importances_
    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values(by='importance', ascending=False)
    
    plot_fi_path = "feature_importance_rf.png"
    fig_fi, ax_fi = plt.subplots(figsize=(10, 8))
    ax_fi.barh(fi_df.head(15)['feature'][::-1], fi_df.head(15)['importance'][::-1], color='rebeccapurple')
    ax_fi.set_title('Top 15 Variáveis - Random Forest')
    plt.savefig(plot_fi_path)
    plt.close()
    mlflow.log_artifact(plot_fi_path)

    print(f"Random Forest registrado! AUC: {metrics_rf['roc_auc']:.4f}")

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
2026/03/29 16:48:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 16:48:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest registrado! AUC: 0.9787


In [24]:
import mlflow
import pandas as pd

# 1. Define o experimento que queremos consultar
experiment_name = "Projeto_Churn_Tech_Challenge"
experiment = mlflow.get_experiment_by_name(experiment_name)

# 2. Busca todas as "runs" (execuções) desse experimento
# Ordenamos pela métrica 'f1_score' para o melhor ficar no topo
df_results = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_score DESC"]
)

# 3. Selecionamos apenas as colunas que importam para o relatório
# Nota: O MLflow coloca 'metrics.' e 'params.' na frente dos nomes
cols_to_show = [
    "tags.mlflow.runName", 
    "params.model_type", 
    "metrics.accuracy", 
    "metrics.precision", 
    "metrics.recall", 
    "metrics.f1_score",
    "metrics.roc_auc"
]

# Filtramos apenas as colunas que existem (evita erro se alguma run falhou)
available_cols = [c for c in cols_to_show if c in df_results.columns]
tabela_final = df_results[available_cols]

# Renomeando para ficar mais bonito no relatório
tabela_final.columns = [c.replace("metrics.", "").replace("params.", "").replace("tags.mlflow.", "") for c in tabela_final.columns]

print("===== COMPARATIVO FINAL DE MODELOS =====")
display(tabela_final) # No Jupyter/Colab ele gera uma tabela formatada

===== COMPARATIVO FINAL DE MODELOS =====


,runName,model_type,accuracy,precision,recall,f1_score,roc_auc
0,regressao_logistica_baseline,LogisticRegression,0.968772,1.000000,0.882353,0.937500,0.980506
1,decision_tree_baseline,DecisionTreeClassifier,0.968062,0.985251,0.893048,0.936886,0.977468
2,regressao_logistica_baseline,LogisticRegression,0.968062,0.996979,0.882353,0.936170,0.979777
3,decision_tree_baseline,DecisionTreeClassifier,0.967353,0.982353,0.893048,0.935574,0.976837
4,random_forest_baseline,RandomForestClassifier,0.949610,0.996721,0.812834,0.895434,0.978697
5,Dummy_Classifier_baseline,DummyClassifier,0.600635,0.262443,0.278886,0.270414,NaN
6,Dummy_Classifier_baseline,DummyClassifier,0.600635,0.262443,0.278886,0.270414,NaN
